# 04: LASSO & Ridge Regularization

**Goal:** Train LASSO (L1) and Ridge (L2) regularized logistic regression for feature selection and comparison.

## Key Deliverable
Top-20 LASSO-selected words with economic interpretation

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load Data & TF-IDF Features

In [ ]:
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# Create TF-IDF features (same as notebook 03)
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text,
    max_features=5000,
    min_df=5,
    max_df=0.95,
)

print(f"Data loaded: {len(df)} bills, {X_tfidf.shape[1]} TF-IDF features")

## Step 2: Train LASSO (L1) Logistic Regression

In [ ]:
# Train LASSO with CV for lambda tuning
lasso_model = model_utils.train_lasso_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
print(f"✓ LASSO model trained")
print(f"Best C (inverse lambda): {lasso_model.C_[0]:.6f}")

## Step 3: Train Ridge (L2) Logistic Regression

In [ ]:
# Train Ridge with CV for lambda tuning
ridge_model = model_utils.train_ridge_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
print(f"✓ Ridge model trained")
print(f"Best C (inverse lambda): {ridge_model.C_[0]:.6f}")

## Step 4: Extract & Visualize Top LASSO Features

In [ ]:
# Get top-20 LASSO features
top_lasso = model_utils.get_top_features_lasso(lasso_model, feature_names, top_n=20)
print("\nTop-20 LASSO Features (by absolute coefficient):")
print(top_lasso)

# Save
top_lasso.to_csv("../results/tables/lasso_top_features.csv", index=False)
print("\nSaved to results/tables/lasso_top_features.csv")

In [ ]:
# Plot LASSO coefficients
viz_utils.plot_feature_coefficients(
    top_lasso,
    output_path="../results/figures/lasso_coefficients.png",
    title="LASSO Feature Coefficients (Top-20)",
    max_features=20
)

## Step 5: Model Comparison

In [ ]:
# Evaluate both models
evaluator = model_utils.ModelEvaluator(random_state=SEED)

lasso_results = evaluator.evaluate_classifier(
    lasso_model, X_tfidf, y, model_name="LASSO Logistic", cv_splits=5
)
ridge_results = evaluator.evaluate_classifier(
    ridge_model, X_tfidf, y, model_name="Ridge Logistic", cv_splits=5
)

# Compare
comparison = pd.DataFrame([lasso_results, ridge_results])
print("\n=== LASSO vs Ridge Comparison ===")
print(comparison[['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean']])

# Save
comparison.to_csv("../results/tables/lasso_ridge_comparison.csv", index=False)

**Next:** Run `05_random_forest.ipynb`